In [5]:
import sys, logging, re, time, urllib.request, os, hashlib
from pathlib import Path
import pandas as pd
import numpy as np
from tqdm import tqdm

sys.path.insert(0, str(Path.cwd().parent))

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')
log = logging.getLogger(__name__)

PROJ_ROOT = Path.cwd().parent
DATA_DIR = PROJ_ROOT / 'data'
FB_DIR = DATA_DIR / 'fitness_browser'
FB_DIR.mkdir(exist_ok=True)

In [6]:
# Complete list of 60 organisms
ALL_ORGS = [
    'acidovorax_3H11', 'azobra', 'Btheta', 'Bifido', 'Brev2', 'BFirm',
    'Burk376', 'Burkholderia_OAS925', 'Bvulgatus_CL09T03C04', 'Caulo',
    'CL21', 'Cup4G11', 'PS', 'Dda3937', 'Ddia6719', 'DdiaME23', 'Dino',
    'DvH', 'Dyella79', 'HerbieS', 'Kang', 'Keio', 'Korea', 'Koxy',
    'Lysobacter_OAE881', 'Magneto', 'Marino', 'Methanococcus_JJ',
    'Methanococcus_S2', 'Miya', 'MR1', 'Mucilaginibacter_YX36', 'MycoTube',
    'Pedo557', 'Phaeo', 'Ponti', 'pseudo1_N1B4', 'pseudo13_GW456_L13',
    'pseudo3_N2E3', 'pseudo5_N2C3_1', 'pseudo6_N2E2', 'psRCH2', 'Putida',
    'PV4', 'RalstoniaBSBF1503', 'RalstoniaGMI1000', 'RalstoniaPSI07',
    'RalstoniaUW163', 'rhodanobacter_10B01', 'rhodanobacter_R12',
    'rhodanobacter_T8', 'RPal_CGA009', 'SB2B', 'Smeli', 'SyringaeB728a',
    'SyringaeB728a_mexBdelta', 'SynE', 'Variovorax_OAS795', 'WCS417',
    'Xantho'
]

In [7]:
# ─── Download via CGI endpoints ────────────────────────────────────
BASE_URL = "https://fit.genomics.lbl.gov"
FILES = {
    "experiments": f"{BASE_URL}/cgi-bin/createExpData.cgi?orgId={{org}}",
    "fitness":     f"{BASE_URL}/cgi-bin/createFitData.cgi?orgId={{org}}",
    "proteins":    f"{BASE_URL}/cgi-bin/orgSeqs.cgi?orgId={{org}}",
}

def download_fitness_data(organisms, output_dir, timeout=30):
    """Download required files using CGI endpoints with browser impersonation."""
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    # Try curl_cffi first (best bot evasion), fall back to plain requests
    try:
        from curl_cffi.requests import Session
        sess = Session(impersonate="chrome131")
        log.info("Using curl_cffi with Chrome impersonation.")
    except ImportError:
        log.info("curl_cffi not installed; using plain requests with browser headers.")
        sess = requests.Session()
        sess.headers.update({
            'User-Agent': 'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 '
                          '(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
            'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
            'Accept-Language': 'en-US,en;q=0.5',
        })

    # Warm up session
    try:
        sess.get(BASE_URL + "/", timeout=timeout)
    except Exception:
        pass

    for org in tqdm(organisms, desc="Downloading"):
        org_dir = output_dir / org
        org_dir.mkdir(exist_ok=True)
        for label, url_template in FILES.items():
            url = url_template.replace("{org}", org)
            filepath = org_dir / f"{org}_{label}.txt"
            if filepath.exists():
                continue
            try:
                r = sess.get(url, timeout=timeout)
                if r.status_code == 200:
                    filepath.write_bytes(r.content)
            except Exception as e:
                log.warning(f"  {org}/{label}: {e}")

# ─── Download all ───────────────────────────────────────────────────
download_fitness_data(ALL_ORGS, FB_DIR)

2026-06-25 06:13:19,838 INFO Using curl_cffi with Chrome impersonation.


Downloading:   0%|          | 0/60 [00:00<?, ?it/s]

Downloading:   2%|▏         | 1/60 [00:02<02:52,  2.92s/it]

Downloading:   3%|▎         | 2/60 [00:05<02:27,  2.54s/it]

Downloading:   5%|▌         | 3/60 [00:12<04:19,  4.55s/it]

...

Downloading:  95%|█████████▌| 57/60 [02:24<00:05,  1.82s/it]

Downloading:  97%|█████████▋| 58/60 [02:26<00:03,  1.95s/it]

Downloading:  98%|█████████▊| 59/60 [02:30<00:02,  2.46s/it]

Downloading: 100%|██████████| 60/60 [02:31<00:00,  2.12s/it]

Downloading: 100%|██████████| 60/60 [02:31<00:00,  2.53s/it]

In [13]:
# ─── Stressor patterns (compact version) ─────────────────────────
STRESSOR_PATTERNS = {
    'Zn':['zinc','zncl2','znso4','zinc chloride','zinc sulfate'],
    'Cu':['copper','cuso4','cucl2','copper sulfate','copper chloride'],
    'Cd':['cadmium','cdcl2','cdso4','cadmium chloride'],
    'Co':['cobalt','cocl2','coso4','cobalt chloride'],
    'Ni':['nickel','nicl2','niso4','nickel chloride','nickel sulfate'],
    'Cr':['chromium','chromate','k2cro4','dichromate','potassium chromate'],
    'As':['arsenate','arsenite','sodium arsenite','sodium arsenate','nah2aso4'],
    'Hg':['mercury','hgcl2','mercury chloride','mercuric chloride'],
    'Pb':['lead','pbno3','lead nitrate','pbcl2'],
    'Mn':['manganese','mncl2','mnso4','manganese chloride'],
    'Fe':['iron','feso4','fecl3','iron limitation','iron depletion','dipyridyl','bipyridyl'],
    'Se':['selenium','selenite','selenate','sodium selenite'],
    'Ag':['silver','agno3','silver nitrate'],
    'Al':['aluminium','aluminum','alcl3','aluminum chloride'],
    'Ampicillin':['ampicillin'], 'Kanamycin':['kanamycin'],
    'Gentamicin':['gentamicin'], 'Rifampicin':['rifampicin','rifampin'],
    'Chloramphenicol':['chloramphenicol'], 'Tetracycline':['tetracycline'],
    'Phosphomycin':['phosphomycin','fosfomycin'], 'Ceftazidime':['ceftazidime'],
    'Polymyxin':['polymyxin','colistin'], 'Ramoplanin':['ramoplanin'],
    'Vancomycin':['vancomycin'], 'Erythromycin':['erythromycin'],
    'Ciprofloxacin':['ciprofloxacin','cipro'], 'Spectinomycin':['spectinomycin'],
    'Streptomycin':['streptomycin'], 'Carbenicillin':['carbenicillin'],
    'Penicillin':['penicillin'], 'Trimethoprim':['trimethoprim'],
    'Novobiocin':['novobiocin'], 'Bacitracin':['bacitracin'],
    'Linezolid':['linezolid'],
    'H2O2':['hydrogen peroxide','h2o2','peroxide'],
    'Paraquat':['paraquat','methyl viologen'],
    'Menadione':['menadione','vitamin k3'], 'Diamide':['diamide'],
    'Nitric oxide':['nitric oxide','no','spermine noate','deta no'],
    'Acid':['acid','acidic','low ph','ph 4','ph 4.5','ph 5','hcl'],
    'Alkaline':['alkaline','high ph','ph 9','ph 10','naoh','koh'],
    'NaCl':['sodium chloride','nacl','salt','salinity'],
    'KCl':['potassium chloride','kcl'], 'Sucrose':['sucrose','osmotic'],
    'Heat':['heat','high temperature','42c','45c','37c'],
    'Cold':['cold','low temperature','4c','10c','15c','16c'],
    'SDS':['sds','sodium dodecyl sulfate','sodium dodecyl sulphate'],
    'EDTA':['edta','ethylenediaminetetraacetic acid'],
    'Ethanol':['ethanol','etoh'],
    'Bile salts':['bile','bile salts','cholate','deoxycholate'],
    'Nitrogen limitation':['nitrogen limit','n limitation','low nitrogen','ammonium limit'],
    'Phosphate limitation':['phosphate limit','p limitation','low phosphate'],
    'Iron limitation':['iron limit','fe limitation','low iron','dipyridyl'],
    'Carbon limitation':['carbon limit','c limitation','low carbon','minimal medium'],
    'Sulfur limitation':['sulfur limit','sulphur limit','low sulfate','low sulphur'],
    'UV':['uv','ultraviolet','uv irradiation'],
    'Desiccation':['desiccation','drying','dry'],
    'Anaerobic':['anaerobic','anoxic','low oxygen','nitrogen atmosphere'],
    'Biofilm':['biofilm'],
}
FITNESS_THRESHOLD = -2.0
MIN_POSITIVES = 30

def parse_one(org_id):
    org_dir = FB_DIR / org_id
    exp_file = org_dir / f"{org_id}_experiments.txt"
    fit_file = org_dir / f"{org_id}_fitness.txt"
    prot_file = org_dir / f"{org_id}_proteins.txt"
    if not exp_file.exists() or not fit_file.exists():
        return None

    exp_df = pd.read_csv(exp_file, sep="\t", low_memory=False)
    fit_df = pd.read_csv(fit_file, sep="\t", low_memory=False)
    fit_cols = set(fit_df.columns)

    # map experiment names → fitness columns
    exp_to_col = {}
    for _, row in exp_df.iterrows():
        name, desc = row['expName'], row['expDesc']
        if name in fit_cols:
            exp_to_col[name] = name
        else:
            for col in fit_cols:
                if col == name + " " + desc or desc in col or col.startswith(name + " "):
                    exp_to_col[name] = col
                    break

    labels = pd.DataFrame({'locusId': fit_df['locusId'], 'organism': org_id})
    for stressor, patterns in STRESSOR_PATTERNS.items():
        matching = [idx for idx, row in exp_df.iterrows()
                    if any(re.search(r'(?i)'+p, str(row['expDesc'])) for p in patterns)]
        cols = [exp_to_col.get(exp_df.loc[idx, 'expName']) for idx in matching]
        cols = [c for c in cols if c is not None]
        labels[stressor] = (
            (fit_df[cols].min(axis=1, skipna=True) < FITNESS_THRESHOLD).fillna(False).astype(int)
            if cols else 0
        )

    if not prot_file.exists():
        return None
    seqs = {}
    with open(prot_file, 'r') as f:
        current_id, lines = None, []
        for line in f:
            line = line.strip()
            if line.startswith('>'):
                if current_id: seqs[current_id] = ''.join(lines)
                current_id = line[1:].split(':')[-1].split()[0]
                lines = []
            elif line: lines.append(line)
        if current_id: seqs[current_id] = ''.join(lines)
    seq_df = pd.DataFrame(list(seqs.items()), columns=['locusId','sequence'])

    # ── Fix: force both locusId columns to string ──
    labels['locusId'] = labels['locusId'].astype(str)
    seq_df['locusId'] = seq_df['locusId'].astype(str)

    labels = labels.merge(seq_df, on='locusId', how='inner')
    return labels

In [14]:
# ─── Parse all organisms ───────────────────────────────────────────
dfs = []
for org in ALL_ORGS:
    df = parse_one(org)
    if df is not None and len(df) > 0:
        dfs.append(df)
    else:
        log.warning(f"Skipping {org} (parsing failed or no proteins)")

if not dfs:
    raise SystemExit("No organisms parsed. Check the download URLs or local files.")

all_data = pd.concat(dfs, ignore_index=True)

In [15]:
# ─── Clean sequences ──────────────────────────────────────────────
all_data = all_data[all_data['sequence'].notna()]
all_data['sequence'] = all_data['sequence'].astype(str)
all_data = all_data[~all_data['sequence'].isin(['nan','None',''])]

In [16]:
# ─── Unique protein IDs ───────────────────────────────────────────
raw_ids = all_data['organism'].astype(str) + '|' + all_data['locusId'].astype(str)
dup = raw_ids.groupby(raw_ids).cumcount()
all_data['protein_id'] = [f"{rid}{'_dup'+str(d) if d else ''}" for rid, d in zip(raw_ids, dup)]

In [17]:
# ─── Active stressors ─────────────────────────────────────────────
stressor_cols = [c for c in all_data.columns if c not in ('locusId','organism','sequence','protein_id')]
stressor_counts = all_data[stressor_cols].sum()
active_stressors = stressor_counts[stressor_counts >= MIN_POSITIVES].index.tolist()
log.info(f"Active stressors: {len(active_stressors)}")

2026-06-25 06:17:50,788 INFO Active stressors: 50


In [18]:
# ─── Final DataFrame ──────────────────────────────────────────────
keep = ['protein_id','sequence','organism'] + active_stressors
labeled_pd = all_data[keep].set_index('protein_id')

In [19]:
# ─── Save ─────────────────────────────────────────────────────────
labeled_pd.to_parquet(DATA_DIR / 'labeled_pd.parquet')
import json
with open(DATA_DIR / 'active_stressors.json', 'w') as f:
    json.dump(active_stressors, f)

log.info(f"Saved {len(labeled_pd)} proteins × {len(active_stressors)} stressors.")

2026-06-25 06:17:54,027 INFO Saved 215051 proteins × 50 stressors.
